# N15v4: 3-Family Stacking (Prior-Corrected)

RealMLP + TabPFN-3 + GBDT(Numeric TE) → blend / L2 meta.

**N14v3 executed:** Blend TabPFN=0.28 + PL → OOF 0.95031 / LB **0.95029** (new organic Era 2 best).

v4: proper newlines + prior-correction on TabPFN/RealMLP + Rule 20 checkpoint loading.

Attach `prior-labsai/tabpfn-3`. GPU T4 x2.


In [ ]:
import os
os.environ["TABPFN_MODEL_CACHE_DIR"]="/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
os.environ["TABPFN_NO_BROWSER"]="1"
os.environ["TABPFN_MAX_BATCHED_TEST_ROWS"]="4096"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
!pip install -q tabpfn pytabkit 2>/dev/null || true


In [ ]:
import os, gc, time, warnings
import numpy as np, pandas as pd, torch, lightgbm as lgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings("ignore")

ID_COL,TARGET_COL="id","health_condition"
N_FOLDS,SEED,N_CLASSES=5,42,3
GPU_ENABLED=torch.cuda.is_available(); N_GPUS=torch.cuda.device_count() if GPU_ENABLED else 0
DEVICE="cuda" if GPU_ENABLED else "cpu"; cb_task="GPU" if GPU_ENABLED else "CPU"
print(f"GPU={GPU_ENABLED} | n_gpus={N_GPUS}")

_KAGGLE="/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
TABPFN_CKPT=f"{_KAGGLE}/tabpfn-v3-classifier-v3_default.ckpt"
os.environ["TABPFN_MODEL_CACHE_DIR"]=_KAGGLE; os.environ["TABPFN_NO_BROWSER"]="1"

TABPFN_OK=False
if os.path.isfile(TABPFN_CKPT):
    from tabpfn import TabPFNClassifier
    TABPFN_OK=True; print(f"TabPFN checkpoint OK: {TABPFN_CKPT}")
else:
    print(f"MISSING: {TABPFN_CKPT}")

REALMLP_OK=False
try:
    from pytabkit import RealMLP_TD_Classifier
    REALMLP_OK=True; print("RealMLP: OK")
except Exception as e:
    print(f"RealMLP unavailable: {e}")

def prior_correct(proba, priors):
    bal=proba/priors; bal/=bal.sum(axis=1,keepdims=True); return bal

def batch_predict(clf,X,bs=4096):
    X=np.asarray(X); return np.vstack([clf.predict_proba(X[i:i+bs]) for i in range(0,len(X),bs)])


In [ ]:
for p in ["/kaggle/input/competitions/playground-series-s6e7","/kaggle/input/playground-series-s6e7","../../data"]:
    if os.path.exists(os.path.join(p,"train.csv")): DATA_DIR=p; break
train_df=pd.read_csv(os.path.join(DATA_DIR,"train.csv"))
test_df=pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
print(f"Train: {train_df.shape} | Test: {test_df.shape}")
le=LabelEncoder(); y=le.fit_transform(train_df[TARGET_COL])
class_priors=np.bincount(y,minlength=N_CLASSES).astype(np.float64)/len(y)
print(f"Class priors: {class_priors}")
cat_cols=["stress_level","physical_activity_level","diet_type","gender","smoking_alcohol","sleep_quality"]
num_cols=["sleep_duration","heart_rate","bmi","calorie_expenditure","step_count","exercise_duration","water_intake"]
all_cols=cat_cols+num_cols
X_str=train_df[all_cols].fillna("__NA__").astype(str); X_str_te=test_df[all_cols].fillna("__NA__").astype(str)
imp=SimpleImputer(strategy="median"); X_num=imp.fit_transform(train_df[num_cols]); X_num_te=imp.transform(test_df[num_cols])
te=TargetEncoder(cv=5,target_type="multiclass",random_state=SEED)
X_te=te.fit_transform(X_str,y); X_te_test=te.transform(X_str_te)
X_full=np.hstack([X_num,X_te]); X_full_test=np.hstack([X_num_te,X_te_test]); print(f"Features: {X_full.shape[1]}")
X_raw=train_df[all_cols].copy(); X_raw_te=test_df[all_cols].copy()
for c in cat_cols:
    X_raw[c]=X_raw[c].fillna("Missing").astype("category"); X_raw_te[c]=X_raw_te[c].fillna("Missing").astype("category")
for c in num_cols:
    med=X_raw[c].median(); X_raw[c]=X_raw[c].fillna(med); X_raw_te[c]=X_raw_te[c].fillna(med)
folds=list(StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED).split(X_full,y))


In [ ]:
print("="*60+"\nFAMILY C: GBDT\n"+"="*60)
t0=time.time(); oof_C=np.zeros((len(train_df),N_CLASSES)); tst_C=np.zeros((len(test_df),N_CLASSES)); SEEDS=[42,2026,999]
for fold,(tr,va) in enumerate(folds):
    print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
    X_tr,X_va=X_full[tr],X_full[va]; y_tr=y[tr]
    sw=compute_sample_weight("balanced",y_tr); cc=np.bincount(y_tr,minlength=N_CLASSES); cb_w=[len(y_tr)/(N_CLASSES*c) for c in cc]
    fv=np.zeros((len(va),N_CLASSES)); ft=np.zeros((len(test_df),N_CLASSES)); nm=3*len(SEEDS)
    for s in SEEDS:
        m=CatBoostClassifier(iterations=1200,learning_rate=0.03,depth=6,class_weights=cb_w,random_seed=s,verbose=0,task_type=cb_task)
        m.fit(X_tr,y_tr,eval_set=(X_va,y[va]),early_stopping_rounds=100)
        fv+=m.predict_proba(X_va)/nm; ft+=m.predict_proba(X_full_test)/nm; del m
        lp=dict(n_estimators=1200,learning_rate=0.03,num_leaves=63,class_weight="balanced",random_state=s,n_jobs=-1,verbose=-1)
        if GPU_ENABLED: lp["device_type"]="gpu"
        m=lgb.LGBMClassifier(**lp)
        m.fit(X_tr,y_tr,eval_set=[(X_va,y[va])],callbacks=[lgb.early_stopping(100,verbose=False)])
        fv+=m.predict_proba(X_va)/nm; ft+=m.predict_proba(X_full_test)/nm; del m
        m=HistGradientBoostingClassifier(max_iter=1200,learning_rate=0.03,max_leaf_nodes=63,class_weight="balanced",random_state=s,early_stopping=True,validation_fraction=0.1,n_iter_no_change=100)
        m.fit(X_tr,y_tr,sample_weight=sw); fv+=m.predict_proba(X_va)/nm; ft+=m.predict_proba(X_full_test)/nm; del m
    oof_C[va]=fv; tst_C+=ft/N_FOLDS; print(f"BAcc={balanced_accuracy_score(y[va],fv.argmax(1)):.5f}"); gc.collect()
score_C=balanced_accuracy_score(y,oof_C.argmax(1)); print(f">>> GBDT OOF: {score_C:.5f} ({time.time()-t0:.0f}s) <<<")


In [ ]:
print("="*60+"\nFAMILY B: TabPFN prior-corr\n"+"="*60)
oof_B=np.zeros((len(train_df),N_CLASSES)); tst_B=np.zeros((len(test_df),N_CLASSES)); score_B=0.0
if TABPFN_OK:
    X_tab=X_num.copy(); X_tab_te=X_num_te.copy()
    for c in cat_cols:
        cats=pd.Categorical(train_df[c].fillna("Missing").astype(str))
        X_tab=np.hstack([X_tab,cats.codes.astype(np.float64).reshape(-1,1)])
        te_codes=pd.Categorical(test_df[c].fillna("Missing").astype(str),categories=cats.categories).codes.astype(np.float64).reshape(-1,1)
        X_tab_te=np.hstack([X_tab_te,te_codes])
    devices=["cuda:0","cuda:1"] if N_GPUS>=2 else DEVICE
    try:
        t0=time.time()
        for fold,(tr,va) in enumerate(folds):
            print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
            sub,_=train_test_split(np.arange(len(tr)),train_size=min(100_000,len(tr)),stratify=y[tr],random_state=SEED+fold)
            clf=TabPFNClassifier(device=devices,n_estimators=2,balance_probabilities=True,fit_mode="fit_with_cache",model_path=TABPFN_CKPT,ignore_pretraining_limits=True)
            clf.fit(X_tab[tr[sub]],y[tr[sub]])
            vp=prior_correct(batch_predict(clf,X_tab[va]),class_priors)
            tp=prior_correct(batch_predict(clf,X_tab_te),class_priors)
            oof_B[va]=vp; tst_B+=tp/N_FOLDS
            print(f"BAcc={balanced_accuracy_score(y[va],vp.argmax(1)):.5f}")
            del clf; gc.collect()
            if GPU_ENABLED: torch.cuda.empty_cache()
        score_B=balanced_accuracy_score(y,oof_B.argmax(1)); print(f">>> TabPFN OOF: {score_B:.5f} ({time.time()-t0:.0f}s) <<<")
    except Exception as e:
        print(f"TabPFN FAILED: {e}"); score_B=0.0
else:
    print("TabPFN skipped")


In [ ]:
print("="*60+"\nFAMILY A: RealMLP prior-corr\n"+"="*60)
oof_A=np.zeros((len(train_df),N_CLASSES)); tst_A=np.zeros((len(test_df),N_CLASSES)); score_A=0.0
if REALMLP_OK:
    try:
        t0=time.time()
        for fold,(tr,va) in enumerate(folds):
            print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
            clf=RealMLP_TD_Classifier(device=DEVICE if GPU_ENABLED else "cpu", random_state=SEED+fold, n_cv=1, verbosity=0)
            clf.fit(X_raw.iloc[tr], y[tr], X_raw.iloc[va], y[va], cat_col_names=cat_cols)
            vp=prior_correct(clf.predict_proba(X_raw.iloc[va]), class_priors)
            tp=prior_correct(clf.predict_proba(X_raw_te), class_priors)
            oof_A[va]=vp; tst_A+=tp/N_FOLDS
            print(f"BAcc={balanced_accuracy_score(y[va],vp.argmax(1)):.5f}")
            del clf; gc.collect()
            if GPU_ENABLED: torch.cuda.empty_cache()
        score_A=balanced_accuracy_score(y,oof_A.argmax(1)); print(f">>> RealMLP OOF: {score_A:.5f} ({time.time()-t0:.0f}s) <<<")
    except Exception as e:
        print(f"RealMLP FAILED: {e}"); score_A=0.0
else:
    print("RealMLP skipped")


In [ ]:
print("="*60+"\nFUSION\n"+"="*60)
avail_oofs,avail_tsts,labels=[],[],[]
if score_A>0.88: avail_oofs.append(oof_A); avail_tsts.append(tst_A); labels.append("RealMLP")
if score_B>0.85: avail_oofs.append(oof_B); avail_tsts.append(tst_B); labels.append("TabPFN")
avail_oofs.append(oof_C); avail_tsts.append(tst_C); labels.append("GBDT")
print("Families:", labels)

def neg_bacc(w):
    w=np.abs(w); w=w/w.sum()
    return -balanced_accuracy_score(y, sum(w[i]*avail_oofs[i] for i in range(len(w))).argmax(1))

res=minimize(neg_bacc, x0=np.ones(len(avail_oofs))/len(avail_oofs), method="Nelder-Mead", options={"maxiter":3000})
opt_w=np.abs(res.x); opt_w=opt_w/opt_w.sum()
opt_oof=sum(opt_w[i]*avail_oofs[i] for i in range(len(opt_w)))
opt_tst=sum(opt_w[i]*avail_tsts[i] for i in range(len(opt_w)))
opt_score=balanced_accuracy_score(y, opt_oof.argmax(1))
print(f"Optimal Blend: {opt_score:.5f} {dict(zip(labels,[round(w,3) for w in opt_w]))}")
simple_oof=np.mean(avail_oofs,axis=0); simple_tst=np.mean(avail_tsts,axis=0)
simple_score=balanced_accuracy_score(y, simple_oof.argmax(1)); print(f"Simple Avg: {simple_score:.5f}")

l2_score=0.0; L2_tst=None
if len(avail_oofs)>=2:
    L1_oof=np.hstack(avail_oofs); L1_tst=np.hstack(avail_tsts)
    L2_oof=np.zeros((len(train_df),N_CLASSES)); L2_tst=np.zeros((len(test_df),N_CLASSES))
    for fold,(tr,va) in enumerate(StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED*3).split(L1_oof,y)):
        lp=dict(n_estimators=500,learning_rate=0.05,num_leaves=15,class_weight="balanced",random_state=SEED,n_jobs=-1,verbose=-1)
        if GPU_ENABLED: lp["device_type"]="gpu"
        meta=lgb.LGBMClassifier(**lp)
        meta.fit(L1_oof[tr],y[tr],eval_set=[(L1_oof[va],y[va])],callbacks=[lgb.early_stopping(50,verbose=False)])
        L2_oof[va]=meta.predict_proba(L1_oof[va]); L2_tst+=meta.predict_proba(L1_tst)/N_FOLDS
    l2_score=balanced_accuracy_score(y,L2_oof.argmax(1)); print(f"L2 Meta: {l2_score:.5f}")

candidates=[("GBDT solo",score_C,tst_C),("Optimal Blend",opt_score,opt_tst),("Simple Avg",simple_score,simple_tst)]
if l2_score>0: candidates.append(("L2 Meta",l2_score,L2_tst))
best_name,best_score,best_probs=max(candidates,key=lambda x:x[1])
print(f">>> BEST: {best_name} | OOF {best_score:.5f} <<<")


In [ ]:
PSEUDO_THRESHOLD=0.99
mask=best_probs.max(1)>=PSEUDO_THRESHOLD; n_pseudo=int(mask.sum())
print(f"Pseudo-labels: {n_pseudo} ({100*n_pseudo/len(test_df):.1f}%)")
if n_pseudo>1000:
    py=best_probs[mask].argmax(1)
    aug_X=np.vstack([X_full,X_full_test[mask]]); aug_y=np.concatenate([y,py])
    aug_tst=np.zeros((len(test_df),N_CLASSES))
    for fold,(tr,va) in enumerate(StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED*5).split(aug_X,aug_y)):
        sw=compute_sample_weight("balanced",aug_y[tr])
        cc=np.bincount(aug_y[tr],minlength=N_CLASSES); cw=[len(aug_y[tr])/(N_CLASSES*c) for c in cc]
        fp=np.zeros((len(test_df),N_CLASSES))
        m=CatBoostClassifier(iterations=1200,learning_rate=0.03,depth=6,class_weights=cw,random_seed=SEED,verbose=0,task_type=cb_task)
        m.fit(aug_X[tr],aug_y[tr]); fp+=m.predict_proba(X_full_test)/3; del m
        lp=dict(n_estimators=1200,learning_rate=0.03,num_leaves=63,class_weight="balanced",random_state=SEED,n_jobs=-1,verbose=-1)
        if GPU_ENABLED: lp["device_type"]="gpu"
        m=lgb.LGBMClassifier(**lp); m.fit(aug_X[tr],aug_y[tr]); fp+=m.predict_proba(X_full_test)/3; del m
        m=HistGradientBoostingClassifier(max_iter=1200,learning_rate=0.03,max_leaf_nodes=63,class_weight="balanced",random_state=SEED)
        m.fit(aug_X[tr],aug_y[tr],sample_weight=sw); fp+=m.predict_proba(X_full_test)/3; del m
        aug_tst+=fp/N_FOLDS
    final_probs=0.7*best_probs+0.3*aug_tst; print("Applied PL 70/30")
else:
    final_probs=best_probs
preds=final_probs.argmax(1)
sub=pd.DataFrame({ID_COL:test_df[ID_COL].astype("int64"),TARGET_COL:le.inverse_transform(preds)})
sub.to_csv("submission.csv",index=False)
print("="*60+"\nN15v4 RESULTS\n"+"="*60)
for name,score in [("RealMLP",score_A),("TabPFN",score_B),("GBDT",score_C)]:
    if score>0: print(f"  {name:12s} {score:.5f}")
print(f"  Selected     {best_name}: {best_score:.5f}")
print("N14v3 LB=0.95029 | N06=0.95011 | v0.9=0.95065")
print(sub[TARGET_COL].value_counts())
